In [ ]:
%pip install pandas
%pip install nltk

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("data\Annotated_Relevance_Set.csv")

# Preview data
df.head()

In [ ]:
%pip install hnswlib

In [20]:
%pip install openai
%pip install --upgrade pydantic pydantic-settings
%pip install --no-cache-dir chromadb
%pip install tqdm


import os
import pandas as pd
import chromadb
from tqdm import tqdm
from openai import AzureOpenAI
from dotenv import load_dotenv  # Import the dotenv package

# Load environment variables from the .env file
load_dotenv()

# Initialize Azure OpenAI client for EPAM Dial API
client = AzureOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),  # Replace with your actual API key
    api_version="2024-02-01",
    azure_endpoint="https://ai-proxy.lab.epam.com"
)

# OpenAI model for embeddings
embedding_model = "text-embedding-ada-002"

# Function to generate embeddings
def get_embedding(text):
    response = client.embeddings.create(
        model=embedding_model,
        input=text
    )
    return response.data[0].embedding  # Extract the embedding vector

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_db")
collection = chroma_client.get_or_create_collection(name="faq_embeddings")

# Load datasets
faq_bank = pd.read_csv("data/FAQ_Bank.csv")
user_queries = pd.read_csv("data/User_Query_Bank.csv")

# Handle NaN values in the dataframes
faq_bank.fillna("", inplace=True)
user_queries.fillna("", inplace=True)

# Embed and store FAQ items
print("Embedding and storing FAQ items...")
for index, row in tqdm(faq_bank.iterrows(), total=len(faq_bank)):
    faq_text = row["question"]  # Adjust column name if needed
    faq_embedding = get_embedding(faq_text)

    if faq_embedding:
        collection.add(
            ids=[f"faq_{index}"],
            embeddings=[faq_embedding],
            metadatas=[{"text": faq_text}]
        )

# Embed and store User Queries
print("Embedding and storing User Queries...")
for index, row in tqdm(user_queries.iterrows(), total=len(user_queries)):
    query_text = row["query"]  # Adjust column name if needed
    query_embedding = get_embedding(query_text)

    if query_embedding:
        collection.add(
            ids=[f"query_{index}"],
            embeddings=[query_embedding],
            metadatas=[{"text": query_text}]
        )

print("All embeddings stored successfully in ChromaDB!")

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


...

Embedding and storing FAQ items...


  0%|          | 0/15919 [00:00<?, ?it/s]Insert of existing embedding ID: faq_0
Add of existing embedding ID: faq_0
  0%|          | 1/15919 [00:01<6:21:50,  1.44s/it]Insert of existing embedding ID: faq_1
Add of existing embedding ID: faq_1
  0%|          | 2/15919 [00:02<4:59:58,  1.13s/it]Insert of existing embedding ID: faq_2
Add of existing embedding ID: faq_2
  0%|          | 3/15919 [00:03<4:10:29,  1.06it/s]Insert of existing embedding ID: faq_3
Add of existing embedding ID: faq_3
  0%|          | 4/15919 [00:03<3:36:36,  1.22it/s]Insert of existing embedding ID: faq_4
Add of existing embedding ID: faq_4
  0%|          | 5/15919 [00:04<3:36:43,  1.22it/s]Insert of existing embedding ID: faq_5
Add of existing embedding ID: faq_5
  0%|          | 6/15919 [00:05<3:42:33,  1.19it/s]Insert of existing embedding ID: faq_6
Add of existing embedding ID: faq_6
  0%|          | 7/15919 [00:06<3:37:55,  1.22it/s]Insert of existing embedding ID: faq_7
Add of existing embedding ID: faq_7
  

Embedding and storing User Queries...


100%|██████████| 1201/1201 [14:02<00:00,  1.43it/s]

All embeddings stored successfully in ChromaDB!


In [13]:
faq_bank = pd.read_csv("data/FAQ_Bank.csv")
user_queries = pd.read_csv("data/User_Query_Bank.csv")

faq_bank.head()

,index,url,question,original answer,answer,answer wLink,source,language,type
0,0,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Does AHCCCS have a centralized resource for m...,"<dd><b>Answer: </b>Yes, the AHCCCS <a href=""/P...","Yes, the AHCCCS Medical Coding Resources webp...","Answer: Yes, the AHCCCS Medical Coding Resourc...",AHCCCS,en,Question
1,1,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Are there billing codes available for COVID-1...,<dd><b>Answer: </b>Yes. The Centers for Medica...,Yes. The Centers for Medicare & Medicaid Serv...,Answer: Yes. The Centers for Medicare & Medica...,AHCCCS,en,Question
2,2,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Will AHCCCS issue guidance regarding prior au...,<dd><b>Answer: </b>Prior authorization is not ...,Prior authorization is not permitted for COVI...,Answer: Prior authorization is not permitted f...,AHCCCS,en,Question
3,3,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Is there a claims modifier for services relat...,"<dd><b>Answer: </b>Yes, AHCCCS has designated ...","Yes, AHCCCS has designated the CR modifier to...","Answer: Yes, AHCCCS has designated the CR modi...",AHCCCS,en,Question
4,4,https://www.azahcccs.gov/AHCCCS/AboutUs/covid1...,Does AHCCCS cover testing for COVID-19?,"<dd><b>Answer: </b>Yes, AHCCCS covers COVID-19...","Yes, AHCCCS covers COVID-19 testing. HCPCS U0...","Answer: Yes, AHCCCS covers COVID-19 testing. H...",AHCCCS,en,Question
